In [1]:
import shutil
from pathlib import Path

import instructor
import lancedb
from google.generativeai import GenerativeModel
from lancedb.rerankers import ColbertReranker

from dreamai.ai import ModelName
from dreamai.prompt import Prompt
from dreamai.prompts.models import TableDescription
from dreamai.rag import add_to_lance_table, pdf_to_md_docs

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
ask_kid = instructor.from_gemini(
    client=GenerativeModel(model_name=ModelName.GEMINI_FLASH)
)

In [3]:
shutil.rmtree("lance")

In [4]:
LANCE_URI = "lance/kid/"

In [5]:
lance_db = lancedb.connect(uri=LANCE_URI)

In [6]:
files = ["soccer.txt", "finance.txt"]
md_data = {Path(file).stem: pdf_to_md_docs(file_path=file) for file in files}

In [7]:
table_description_prompt = Prompt.load("dreamai/prompts/table_description_prompt.json")
table_descriptions = []
for file in files:
    name = Path(file).stem
    table_descriptions.append(
        TableDescription(
            name=name,
            description=ask_kid.create(
                response_model=str,
                **table_description_prompt.gemini_kwargs(
                    database_name=name, sample_text=md_data[name].markdown
                ),  # type: ignore
            ),
        )
    )

In [ ]:
# table = add_to_lance_table(db=lance_db, table_name="soccer", data=md_docs.chunks)
# reranker = ColbertReranker("answerdotai/answerai-colbert-small-v1")
# res = table.search(query=query, query_type="hybrid").rerank(reranker=reranker)

In [13]:
print(table_descriptions[0].description)

This database provides an extensive guide to soccer, encompassing its history, rules, tactics, and key figures. It includes detailed explanations of basic rules, advanced concepts like formations and set pieces, and a glossary of common soccer terms. The text explores the cultural significance of soccer as a global sport and highlights its dynamic strategic aspects.


In [14]:
print(table_descriptions[1].description)

This database provides comprehensive information on personal finance management, covering topics like budgeting, saving, investing, insurance, tax planning, retirement planning, and estate planning. The text includes detailed explanations of various financial concepts, strategies, and best practices, suitable for individuals seeking a comprehensive guide to managing their finances.
